In [19]:
# Load dataset

import pandas as pd

df = pd.read_csv("preprocessed_dataset.csv")

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully
Dataset shape: (287421, 12)


,id,title,category,category_code,published_date,updated_date,authors,first_author,summary,summary_word_count,combined_text,cleaned_text
0,cs-9308101v1,Dynamic Backtracking,Artificial Intelligence,cs.AI,1993-08-01,1993-08-01,['M. L. Ginsberg'],'M. L. Ginsberg',Because of their occasional need to return to ...,79,dynamic backtracking occasional need return sh...,dynamic backtrack occasional need return shall...
1,cs-9308102v1,A Market-Oriented Programming Environment and ...,Artificial Intelligence,cs.AI,1993-08-01,1993-08-01,['M. P. Wellman'],'M. P. Wellman',Market price systems constitute a well-underst...,119,market oriented programming environment applic...,market orient program environment application ...
2,cs-9309101v1,An Empirical Analysis of Search in GSAT,Artificial Intelligence,cs.AI,1993-09-01,1993-09-01,"['I. P. Gent', 'T. Walsh']",'I. P. Gent',We describe an extensive study of search in GS...,167,empirical analysis search gsat describe extens...,empirical analysis search gsat describe extens...
3,cs-9311101v1,The Difficulties of Learning Logic Programs wi...,Artificial Intelligence,cs.AI,1993-11-01,1993-11-01,"['F. Bergadano', 'D. Gunetti', 'U. Trinchero']",'F. Bergadano',As real logic programmers normally use cut (!)...,174,difficulties learning logic programs cut real ...,difficulty learn logic program cut real logic ...
4,cs-9311102v1,Software Agents: Completing Patterns and Const...,Artificial Intelligence,cs.AI,1993-11-01,1993-11-01,"['J. C. Schlimmer', 'L. A. Hermens']",'J. C. Schlimmer',To support the goal of allowing users to recor...,187,software agents completing patterns constructi...,software agent complete pattern construct user...


In [20]:
# Clean and prepare text

df = df.dropna(subset=["cleaned_text", "title", "summary"]).copy()

df["cleaned_text"] = df["cleaned_text"].astype(str)
df["title"] = df["title"].astype(str)
df["summary"] = df["summary"].astype(str)

df = df[df["cleaned_text"].str.strip() != ""].copy()

print("Text prepared successfully")
print("Dataset shape after cleaning:", df.shape)

print("\nMissing values:")
print(df[["cleaned_text", "title", "summary"]].isnull().sum())

display(df[["title", "category", "summary", "cleaned_text"]].head())

Text prepared successfully
Dataset shape after cleaning: (287421, 12)

Missing values:
cleaned_text    0
title           0
summary         0
dtype: int64


,title,category,summary,cleaned_text
0,Dynamic Backtracking,Artificial Intelligence,Because of their occasional need to return to ...,dynamic backtrack occasional need return shall...
1,A Market-Oriented Programming Environment and ...,Artificial Intelligence,Market price systems constitute a well-underst...,market orient program environment application ...
2,An Empirical Analysis of Search in GSAT,Artificial Intelligence,We describe an extensive study of search in GS...,empirical analysis search gsat describe extens...
3,The Difficulties of Learning Logic Programs wi...,Artificial Intelligence,As real logic programmers normally use cut (!)...,difficulty learn logic program cut real logic ...
4,Software Agents: Completing Patterns and Const...,Artificial Intelligence,To support the goal of allowing users to recor...,software agent complete pattern construct user...


In [21]:
# Create TF-IDF matrix

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,
    min_df=3,
    max_df=0.85
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df["cleaned_text"])

print("TF-IDF matrix created successfully")
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix created successfully
TF-IDF matrix shape: (287421, 10000)


In [22]:
# Build recommendation system with improved query preprocessing

import re
import nltk

from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

# Download the lemmatization resources
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

lemmatizer = WordNetLemmatizer()


def clean_user_query(query):
    """
    Apply preprocessing similar to the cleaned dataset text.
    """

    query = str(query).lower()

    # Remove punctuation and special characters
    query = re.sub(r"[^a-zA-Z0-9\s]", " ", query)

    # Remove repeated spaces
    query = re.sub(r"\s+", " ", query).strip()

    words = query.split()

    # Remove English stop words
    words = [
        word
        for word in words
        if word not in ENGLISH_STOP_WORDS
    ]

    # Lemmatize words as verbs
    try:
        words = [
            lemmatizer.lemmatize(word, pos="v")
            for word in words
        ]
    except LookupError:
        # Continue without lemmatization if the resource is unavailable
        pass

    return " ".join(words)


def recommend_papers(user_query, top_n=5):
    """
    Recommend research papers using TF-IDF cosine similarity.
    """

    cleaned_query = clean_user_query(user_query)

    if cleaned_query == "":
        raise ValueError(
            "The query is empty after preprocessing. "
            "Please enter a more descriptive query."
        )

    query_vector = tfidf_vectorizer.transform(
        [cleaned_query]
    )

    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    top_n = min(top_n, len(df))

    top_indices = (
        similarity_scores
        .argsort()[::-1][:top_n]
    )

    results = df.iloc[top_indices].copy()

    results["similarity_score"] = (
        similarity_scores[top_indices]
    )

    columns_to_show = [
        column
        for column in [
            "title",
            "category",
            "authors",
            "summary",
            "similarity_score"
        ]
        if column in results.columns
    ]

    return results[columns_to_show]

In [23]:
# Test recommendation system

query = "machine learning for medical diagnosis"

recommendations = recommend_papers(query, top_n=5)

print("User query:", query)
display(recommendations)

User query: machine learning for medical diagnosis


,title,category,authors,summary,similarity_score
175115,Sim4Seg: Boosting Multimodal Multi-disease Med...,Computer Vision and Pattern Recognition,"['Lingran Song', 'Yucheng Zhou', 'Jianbing Shen']",Despite significant progress in pixel-level me...,0.594807
170531,Diagnosing Shoulder Disorders Using Multimodal...,Computer Vision and Pattern Recognition,"['Jindong Hong', 'Wencheng Zhang', 'Shiqin Qia...","Shoulder disorders, such as frozen shoulder (a...",0.535559
22709,Medical Dialogue Generation via Intuitive-then...,Computation and Language (Natural Language Pro...,"['Kaishuai Xu', 'Wenjun Hou', 'Yi Cheng', 'Jia...",Medical dialogue systems have attracted growin...,0.529770
38826,Gradient Boosting Decision Trees on Medical Di...,Machine Learning,"['A. Yarkın Yıldız', 'Asli Kalayci']",Medical diagnosis is a crucial task in the med...,0.512541
106661,Medical Diagnosis with a Novel SVM-CoDOA Based...,Neural and Evolutionary Computing,['M. Hanefi Calp'],Machine Learning is an important sub-field of ...,0.499128


In [24]:
# Evaluate recommendation system using category-based Precision@5

recommendation_test_cases = [
    {
        "query": "computer vision and image recognition",
        "accepted_category_terms": [
            "Computer Vision"
        ]
    },
    {
        "query": "natural language processing",
        "accepted_category_terms": [
            "Computation and Language",
            "Natural Language Processing"
        ]
    },
    {
        "query": "robotics and control systems",
        "accepted_category_terms": [
            "Robotics"
        ]
    },
    {
        "query": "deep learning neural networks",
        "accepted_category_terms": [
            "Machine Learning",
            "Neural and Evolutionary Computing"
        ]
    }
]


def category_is_relevant(category, accepted_terms):
    """
    Check whether the recommended category matches
    one of the expected category terms.
    """

    category = str(category).lower()

    return any(
        term.lower() in category
        for term in accepted_terms
    )


recommendation_evaluation_rows = []

for test_case in recommendation_test_cases:

    query = test_case["query"]
    accepted_terms = test_case["accepted_category_terms"]

    recommendations = recommend_papers(
        query,
        top_n=5
    )

    relevance_results = recommendations[
        "category"
    ].apply(
        lambda category: category_is_relevant(
            category,
            accepted_terms
        )
    )

    relevant_count = int(
        relevance_results.sum()
    )

    precision_at_5 = relevant_count / 5

    recommendation_evaluation_rows.append({
        "Query": query,
        "Expected Category": ", ".join(accepted_terms),
        "Relevant Recommendations": relevant_count,
        "Total Recommendations": 5,
        "Precision@5": precision_at_5
    })


recommendation_evaluation = pd.DataFrame(
    recommendation_evaluation_rows
)

mean_precision_at_5 = (
    recommendation_evaluation["Precision@5"].mean()
)

print("Recommendation Evaluation:")
display(
    recommendation_evaluation.round(4)
)

print(
    "\nMean Precision@5:",
    round(mean_precision_at_5, 4)
)

Recommendation Evaluation:


,Query,Expected Category,Relevant Recommendations,Total Recommendations,Precision@5
0,computer vision and image recognition,Computer Vision,5,5,1.0
1,natural language processing,"Computation and Language, Natural Language Pro...",3,5,0.6
2,robotics and control systems,Robotics,4,5,0.8
3,deep learning neural networks,"Machine Learning, Neural and Evolutionary Comp...",4,5,0.8



Mean Precision@5: 0.8


In [25]:


def split_summary_into_sentences(summary):
    """
    Split a research-paper summary into sentences.
    """

    summary = str(summary)

    sentences = re.split(
        r"(?<=[.!?])\s+",
        summary
    )

    # Remove empty or extremely short sentences
    sentences = [
        sentence.strip()
        for sentence in sentences
        if len(sentence.strip().split()) >= 5
    ]

    return sentences


def answer_question(
    user_question,
    top_n=3,
    candidate_n=15
):
    """
    Retrieve relevant papers and return the sentence
    most closely related to the question.
    """

    cleaned_question = clean_user_query(
        user_question
    )

    if cleaned_question == "":
        raise ValueError(
            "The question is empty after preprocessing."
        )

    question_vector = tfidf_vectorizer.transform(
        [cleaned_question]
    )

    # Compare the question with all research papers
    document_similarity_scores = cosine_similarity(
        question_vector,
        tfidf_matrix
    ).flatten()

    candidate_n = min(
        candidate_n,
        len(df)
    )

    candidate_indices = (
        document_similarity_scores
        .argsort()[::-1][:candidate_n]
    )

    answers = []

    for index in candidate_indices:

        paper = df.iloc[index]
        summary = paper["summary"]

        sentences = split_summary_into_sentences(
            summary
        )

        if not sentences:
            sentences = [str(summary)]

        cleaned_sentences = [
            clean_user_query(sentence)
            for sentence in sentences
        ]

        valid_positions = [
            position
            for position, sentence
            in enumerate(cleaned_sentences)
            if sentence != ""
        ]

        if not valid_positions:
            continue

        valid_cleaned_sentences = [
            cleaned_sentences[position]
            for position in valid_positions
        ]

        sentence_vectors = tfidf_vectorizer.transform(
            valid_cleaned_sentences
        )

        sentence_scores = cosine_similarity(
            question_vector,
            sentence_vectors
        ).flatten()

        best_valid_position = int(
            sentence_scores.argmax()
        )

        original_sentence_position = (
            valid_positions[best_valid_position]
        )

        best_sentence = sentences[
            original_sentence_position
        ]

        sentence_similarity = float(
            sentence_scores[best_valid_position]
        )

        document_similarity = float(
            document_similarity_scores[index]
        )

        combined_score = (
            document_similarity
            + sentence_similarity
        ) / 2

        answers.append({
            "question": user_question,
            "title": paper["title"],
            "category": (
                paper["category"]
                if "category" in df.columns
                else "Not available"
            ),
            "retrieved_answer": best_sentence,
            "document_similarity": document_similarity,
            "sentence_similarity": sentence_similarity,
            "combined_score": combined_score
        })

    answer_results = pd.DataFrame(answers)

    if answer_results.empty:
        return answer_results

    answer_results = (
        answer_results
        .sort_values(
            by="combined_score",
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    return answer_results

In [26]:
# Test QA system

question = "What is deep learning used for?"

qa_results = answer_question(question, top_n=3)

print("Question:", question)
display(qa_results)

Question: What is deep learning used for?


,question,title,category,retrieved_answer,document_similarity,sentence_similarity,combined_score
0,What is deep learning used for?,Why & When Deep Learning Works: Looking Inside...,Machine Learning,We have asked six leading ICRI-CI Deep\nLearni...,0.547484,0.517714,0.532599
1,What is deep learning used for?,Structure preserving deep learning,Machine Learning,A\nlarge amount of progress made in deep learn...,0.485079,0.568789,0.526934
2,What is deep learning used for?,Accelerating Deep Learning with Shrinkage and ...,Machine Learning,Deep Learning is a very powerful machine learn...,0.512319,0.532980,0.522649


In [27]:
# Evaluate QA answer relevance using expected keywords

qa_test_cases = [
    {
        "question": "What is natural language processing?",
        "expected_keywords": [
            "language",
            "text",
            "processing"
        ]
    },
    {
        "question": "What is computer vision?",
        "expected_keywords": [
            "image",
            "visual",
            "vision"
        ]
    },
    {
        "question": "What is deep learning used for?",
        "expected_keywords": [
            "learning",
            "neural",
            "network"
        ]
    },
    {
        "question": "How is machine learning used in prediction?",
        "expected_keywords": [
            "prediction",
            "model",
            "data"
        ]
    },
    {
        "question": "How are neural networks used in research?",
        "expected_keywords": [
            "network",
            "learning",
            "model"
        ]
    }
]


qa_evaluation_rows = []

for test_case in qa_test_cases:

    question = test_case["question"]
    expected_keywords = test_case["expected_keywords"]

    qa_result = answer_question(
        question,
        top_n=1
    )

    if qa_result.empty:
        retrieved_answer = ""
        source_title = "No result"
        matched_keywords = []
    else:
        retrieved_answer = qa_result.iloc[0][
            "retrieved_answer"
        ]

        source_title = qa_result.iloc[0][
            "title"
        ]

        answer_lower = retrieved_answer.lower()

        matched_keywords = [
            keyword
            for keyword in expected_keywords
            if keyword.lower() in answer_lower
        ]

    keyword_relevance_score = (
        len(matched_keywords)
        / len(expected_keywords)
    )

    qa_evaluation_rows.append({
        "Question": question,
        "Source Paper": source_title,
        "Expected Keywords": ", ".join(
            expected_keywords
        ),
        "Matched Keywords": ", ".join(
            matched_keywords
        ),
        "Keyword Relevance Score":
            keyword_relevance_score,
        "Retrieved Answer": retrieved_answer
    })


qa_evaluation = pd.DataFrame(
    qa_evaluation_rows
)

average_qa_keyword_score = (
    qa_evaluation[
        "Keyword Relevance Score"
    ].mean()
)

print("QA Evaluation:")
display(
    qa_evaluation.round(4)
)

print(
    "\nAverage QA Keyword Relevance Score:",
    round(average_qa_keyword_score, 4)
)

QA Evaluation:


,Question,Source Paper,Expected Keywords,Matched Keywords,Keyword Relevance Score,Retrieved Answer
0,What is natural language processing?,Natural Language Processing using Hadoop and K...,"language, text, processing","language, processing",0.6667,Hadoop is one of the\nplatforms that can proce...
1,What is computer vision?,Deep Learning vs. Traditional Computer Vision,"image, visual, vision",vision,0.3333,The paper will also explore how the two sides ...
2,What is deep learning used for?,Why & When Deep Learning Works: Looking Inside...,"learning, neural, network","learning, network",0.6667,We have asked six leading ICRI-CI Deep\nLearni...
3,How is machine learning used in prediction?,Adversarial Debiasing for Unbiased Parameter R...,"prediction, model, data","prediction, model",0.6667,"However, prediction errors from machine learni..."
4,How are neural networks used in research?,Making Neural Networks FAIR,"network, learning, model",network,0.3333,"As such, neural networks themselves have becom..."



Average QA Keyword Relevance Score: 0.5333


In [28]:
# Summarize recommendation and QA evaluation results

evaluation_summary = pd.DataFrame({
    "System Component": [
        "Recommendation System",
        "QA System"
    ],
    "Evaluation Metric": [
        "Mean Precision@5",
        "Average Keyword Relevance Score"
    ],
    "Score": [
        mean_precision_at_5,
        average_qa_keyword_score
    ]
})

print("Recommendation and QA Evaluation Summary:")
display(
    evaluation_summary.round(4)
)

Recommendation and QA Evaluation Summary:


,System Component,Evaluation Metric,Score
0,Recommendation System,Mean Precision@5,0.8000
1,QA System,Average Keyword Relevance Score,0.5333


In [29]:
# Check saved files

import os

files = [
    "recommendation_dataset.csv",
    "recommendation_tfidf_matrix.npz",
    "recommendation_tfidf_vectorizer.pkl"
]

for file in files:
    print(file, "exists:", os.path.exists(file))

recommendation_dataset.csv exists: True
recommendation_tfidf_matrix.npz exists: True
recommendation_tfidf_vectorizer.pkl exists: True
